# Day 19: Multi-GPU Instances & Multi-Instance GPU (MIG)
> *Inference Engineering* — Chapter 3.3 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 17 (GPU Architecture), Day 18 (GPU Generations)


## Concept Overview

Multi-Instance GPU (MIG) partitions a single A100/H100 into up to 7 isolated GPU instances, each with dedicated compute engines, memory, and memory bandwidth. Unlike time-sharing (MPS), MIG provides hard isolation — one instance cannot see another's memory or affect its performance. For inference, MIG enables multi-tenancy: running multiple smaller models on one GPU with guaranteed quality-of-service, or splitting a GPU for different request types (e.g., interactive vs batch). The tradeoff: smaller instances have less bandwidth and less total memory, limiting what models you can serve.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'Total memory: {props.total_memory/1e9:.1f} GB')


## 1. MIG Instance Types (H100)

H100 supports 7 MIG instance sizes, each with different compute and memory allocations.


In [ ]:
# H100 80GB MIG profiles
mig_profiles = [
    # (profile_name, gpu_slices, memory_gb, memory_bw_gb_s, fp16_tflops)
    ('7g.80gb',  7, 79.5, 3350, 989),   # full GPU
    ('4g.40gb',  4, 40.0, 1900, 495),
    ('3g.40gb',  3, 39.5, 1900, 371),
    ('2g.20gb',  2, 19.5, 950,  247),
    ('1g.20gb',  1, 19.5, 475,  124),
    ('1g.10gb',  1, 9.75, 475,  124),
    ('1g.5gb',   1, 4.75, 475,  124),
]

print('H100 MIG Instance Profiles:')
print(f'{"Profile":<12} {"Slices":>8} {"Memory":>10} {"BW (GB/s)":>12} {"FP16 TF":>10} {"Max insts":>10}')
print('-' * 64)
for name, slices, mem, bw, tflops in mig_profiles:
    max_instances = 7 // slices
    print(f'{name:<12} {slices:>8} {mem:>9.1f}G {bw:>12} {tflops:>10} {max_instances:>10}')


## 2. Multi-Tenancy Use Cases

MIG enables running multiple workloads with strict isolation — each gets guaranteed resources.


In [ ]:
# Model what fits in each MIG instance
models = [
    # (name, params_b, fp16_gb, min_bw_needed_gb_s)
    ('GPT-2 117M',    0.117,  0.23, 10),
    ('Llama-3.2-1B',  1.0,   2.0,  20),
    ('Llama-3.2-3B',  3.0,   6.0,  50),
    ('Llama-3-8B',    8.0,   16.0, 80),
    ('Llama-3-70B',  70.0,  140.0, 400),
]

print('Model-to-MIG-Profile Mapping (can it fit + have sufficient BW?):')
print(f'{"Model":<20}', end='')
for name, _, _, _ in mig_profiles[:5]:
    print(f' {name:>10}', end='')
print()
print('-' * 70)
for mname, params, fp16_gb, min_bw in models:
    print(f'{mname:<20}', end='')
    for pname, slices, mem, bw, tflops in mig_profiles[:5]:
        fits = '✓' if fp16_gb <= mem and bw >= min_bw else ('~' if fp16_gb <= mem else '✗')
        print(f' {fits:>10}', end='')
    print()
print('\n✓ = fits + BW ok | ~ = fits, BW marginal | ✗ = OOM')


## 3. MIG vs Time-Sharing vs Dedicated

Comparing isolation, efficiency, and overhead for different GPU sharing strategies.


In [ ]:
strategies = {
    'Dedicated GPU': {
        'isolation': 'Full',
        'overhead': 'None',
        'latency_predictability': 'High',
        'utilization_when_idle': 'Low',
        'memory_isolation': True,
        'best_for': 'Latency-sensitive production models',
    },
    'MIG': {
        'isolation': 'Hardware',
        'overhead': 'None',
        'latency_predictability': 'High',
        'utilization_when_idle': 'Low (per instance)',
        'memory_isolation': True,
        'best_for': 'Multiple models, guaranteed QoS',
    },
    'MPS (Multi-Process)': {
        'isolation': 'Software',
        'overhead': 'Low',
        'latency_predictability': 'Medium',
        'utilization_when_idle': 'Medium',
        'memory_isolation': False,
        'best_for': 'Batch workloads, known-safe tenants',
    },
    'Time-sharing': {
        'isolation': 'Time',
        'overhead': 'Context switch',
        'latency_predictability': 'Low',
        'utilization_when_idle': 'High',
        'memory_isolation': False,
        'best_for': 'Dev/test, non-latency-sensitive',
    },
}
for name, props in strategies.items():
    print(f'{name}:')
    for k, v in props.items():
        print(f'  {k:<30} {v}')
    print()


## Experiments: Try These

1. **MIG partitioning**: On spark-01, run `sudo nvidia-smi mig -cgi 2g.20gb,2g.20gb,2g.20gb -C` and profile inference latency at each partition size.
2. **Isolation test**: Run two models simultaneously in different MIG instances. Verify that one model's memory usage doesn't affect the other's latency.
3. **Utilization modeling**: For a given request rate (req/s) and model config, calculate the optimal MIG profile that minimizes cost while meeting P95 latency SLO.


## Key Takeaways

- MIG partitions a physical GPU into up to 7 isolated instances with dedicated compute, memory, and bandwidth — hard isolation, not time-sharing.
- Each MIG slice gets 1/N of the GPU's resources; a 1g.10gb instance gets ~1/7th of H100's compute and ~1/7th of its bandwidth.
- MIG is ideal for multi-tenant serving of different models with latency SLOs — each tenant gets guaranteed resources.
- The decision between MIG, MPS, and dedicated GPUs depends on isolation requirements, model sizes, and traffic predictability.

## References
- *Inference Engineering* Ch 3.3 — Philip Kiely, Baseten Books 2026
- NVIDIA Multi-Instance GPU User Guide
- NVIDIA A100/H100 MIG documentation
